In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
%sql
use catalog skylens_macro;
use schema team1;

In [0]:
flights_df=spark.readStream.table('skylens_macro.team1.bronze_flights')
airlines_df=spark.read.table('skylens_macro.team1.bronze_airlines')
airports_df=spark.read.table('skylens_macro.team1.bronze_airports')

In [0]:
airlines_df.show(truncate=False)

### Helper function

In [0]:
def parse_hhmm(col_name):
    return lpad(col(col_name).cast("string"), 4, "0")

### TRANSFORM FLIGHTS

In [0]:

# -- Purana code


# df = flights_df \
#    .withColumn("sched_dep_str",
#     when(col("SCHEDULED_DEPARTURE").cast("int") == 2400, lit("0000"))
#     .otherwise(lpad(col("SCHEDULED_DEPARTURE").cast("string"), 4, "0"))
# )\
#     .withColumn("dep_str", parse_hhmm("DEPARTURE_TIME")) \
#     .withColumn("sched_arr_str", parse_hhmm("SCHEDULED_ARRIVAL")) \
#     .withColumn("arr_str", parse_hhmm("ARRIVAL_TIME"))

# # Timestamp conversion
# df = df.withColumn(
#     "scheduled_departure_ts",
#     to_timestamp(concat_ws(" ",
#         concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#         concat_ws(":", substring("sched_dep_str",1,2), substring("sched_dep_str",3,2))
#     ))
# ).withColumn(
#     "actual_departure_ts",
#     to_timestamp(concat_ws(" ",
#         concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#         concat_ws(":", substring("dep_str",1,2), substring("dep_str",3,2))
#     ))
# ).withColumn(
#     "scheduled_arrival_ts",
#     to_timestamp(concat_ws(" ",
#         concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#         concat_ws(":", substring("sched_arr_str",1,2), substring("sched_arr_str",3,2))
#     ))
# ).withColumn(
#     "actual_arrival_ts",
#     to_timestamp(concat_ws(" ",
#         concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#         concat_ws(":", substring("arr_str",1,2), substring("arr_str",3,2))
#     ))
# )

# # Derived columns
# df = df.withColumn("flight_date", to_date(concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),"yyyy-M-d")) \
#        .withColumn("departure_hour", hour("scheduled_departure_ts")) \
#        .withColumn("arrival_hour", hour("scheduled_arrival_ts")) \
#        .withColumn("is_weekend", col("DAY_OF_WEEK").isin([6,7])) \
#        .withColumn("flight_quarter", concat(lit("Q"), quarter("flight_date")))


In [0]:
from pyspark.sql.functions import (
    col, to_date, to_timestamp, concat_ws, concat,
    substring, lpad, lit, hour, quarter, when, coalesce, trim
)

# Step 1 — Fix all time strings inline (handle null/empty too)
def clean_time(c):
    return when(col(c).isNull() | (trim(col(c)) == ""), lit("0000")) \
           .when(col(c).cast("int") == 2400, lit("0000")) \
           .otherwise(lpad(col(c).cast("string"), 4, "0"))

df = flights_df \
    .withColumn("sched_dep_str", clean_time("SCHEDULED_DEPARTURE")) \
    .withColumn("dep_str",       clean_time("DEPARTURE_TIME")) \
    .withColumn("sched_arr_str", clean_time("SCHEDULED_ARRIVAL")) \
    .withColumn("arr_str",       clean_time("ARRIVAL_TIME"))

# Step 2 — Build timestamps safely
def build_ts(time_str_col):
    date_str = concat_ws("-", col("YEAR"), col("MONTH"), col("DAY"))
    full_str = concat_ws(" ", date_str,
                   concat_ws(":", substring(col(time_str_col), 1, 2),
                                  substring(col(time_str_col), 3, 2)))
    return coalesce(
        to_timestamp(full_str, "yyyy-M-d HH:mm"),
        to_timestamp(full_str, "yyyy-MM-dd HH:mm"),
        to_timestamp(full_str, "yyyy-M-d H:mm"),
        to_timestamp(full_str, "yyyy-MM-dd H:mm")
    )

df = df \
    .withColumn("scheduled_departure_ts", build_ts("sched_dep_str")) \
    .withColumn("actual_departure_ts",    build_ts("dep_str")) \
    .withColumn("scheduled_arrival_ts",   build_ts("sched_arr_str")) \
    .withColumn("actual_arrival_ts",      build_ts("arr_str"))

# Step 3 — Derived columns
df = df \
    .withColumn("flight_date",
        to_date(concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")), "yyyy-M-d")
    ) \
    .withColumn("departure_hour", hour("scheduled_departure_ts")) \
    .withColumn("arrival_hour",   hour("scheduled_arrival_ts")) \
    .withColumn("is_weekend",     col("DAY_OF_WEEK").isin([6, 7])) \
    .withColumn("flight_quarter", concat(lit("Q"), quarter("flight_date")))

df.show(5)

In [0]:
# from pyspark.sql.functions import (
#     col, to_date, try_to_timestamp, concat_ws, concat,
#     substring, lpad, lit, hour, quarter, when
# )

# # Step 1 — Parse & fix time strings inline (no UDF)
# df = flights_df \
#     .withColumn("sched_dep_str",
#         when(col("SCHEDULED_DEPARTURE").cast("int") == 2400, lit("0000"))
#         .otherwise(lpad(col("SCHEDULED_DEPARTURE").cast("string"), 4, "0"))
#     ) \
#     .withColumn("dep_str",
#         when(col("DEPARTURE_TIME").cast("int") == 2400, lit("0000"))
#         .otherwise(lpad(col("DEPARTURE_TIME").cast("string"), 4, "0"))
#     ) \
#     .withColumn("sched_arr_str",
#         when(col("SCHEDULED_ARRIVAL").cast("int") == 2400, lit("0000"))
#         .otherwise(lpad(col("SCHEDULED_ARRIVAL").cast("string"), 4, "0"))
#     ) \
#     .withColumn("arr_str",
#         when(col("ARRIVAL_TIME").cast("int") == 2400, lit("0000"))
#         .otherwise(lpad(col("ARRIVAL_TIME").cast("string"), 4, "0"))
#     )

# # Step 2 — Build timestamps
# df = df \
#     .withColumn("scheduled_departure_ts",
#         try_to_timestamp(concat_ws(" ",
#             concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#             concat_ws(":", substring(col("sched_dep_str"), 1, 2), substring(col("sched_dep_str"), 3, 2))
#         ), "yyyy-M-d HH:mm")
#     ) \
#     .withColumn("actual_departure_ts",
#         try_to_timestamp(concat_ws(" ",
#             concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#             concat_ws(":", substring(col("dep_str"), 1, 2), substring(col("dep_str"), 3, 2))
#         ), "yyyy-M-d HH:mm")
#     ) \
#     .withColumn("scheduled_arrival_ts",
#         try_to_timestamp(concat_ws(" ",
#             concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#             concat_ws(":", substring(col("sched_arr_str"), 1, 2), substring(col("sched_arr_str"), 3, 2))
#         ), "yyyy-M-d HH:mm")
#     ) \
#     .withColumn("actual_arrival_ts",
#         try_to_timestamp(concat_ws(" ",
#             concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")),
#             concat_ws(":", substring(col("arr_str"), 1, 2), substring(col("arr_str"), 3, 2))
#         ), "yyyy-M-d HH:mm")
#     )

# # Step 3 — Derived columns
# df = df \
#     .withColumn("flight_date",
#         to_date(concat_ws("-", col("YEAR"), col("MONTH"), col("DAY")), "yyyy-M-d")
#     ) \
#     .withColumn("departure_hour", hour("scheduled_departure_ts")) \
#     .withColumn("arrival_hour",   hour("scheduled_arrival_ts")) \
#     .withColumn("is_weekend",     col("DAY_OF_WEEK").isin([6, 7])) \
#     .withColumn("flight_quarter", concat(lit("Q"), quarter("flight_date")))

# df.show(5)

### AIRPORT STANDARDIZATION FLAG

In [0]:
airport_lookup = airports_df.select(col("IATA_CODE").alias("iata"))

df = df.join(
    airport_lookup.withColumnRenamed("iata", "origin_match"),
    df.ORIGIN_AIRPORT == col("origin_match"),
    "left"
).join(
    airport_lookup.withColumnRenamed("iata", "dest_match"),
    df.DESTINATION_AIRPORT == col("dest_match"),
    "left"
)

df = df.withColumn(
    "is_unknown_airport",
    when(col("origin_match").isNull() | col("dest_match").isNull(), True).otherwise(False)
).drop("origin_match", "dest_match")

### DELAY CATEGORY

In [0]:
df = df.withColumn(
    "delay_category",
    when(col("CANCELLED") == 1, "Cancelled")
    .when(col("ARRIVAL_DELAY") <= 0, "On_Time")
    .when(col("ARRIVAL_DELAY") <= 15, "Slight")
    .when(col("ARRIVAL_DELAY") <= 60, "Moderate")
    .otherwise("severe")
)

### CANCELLATION MAPPING

In [0]:
df = df.withColumn(
    "cancellation_reason_mapped",
    when(col("CANCELLATION_REASON") == "A", "Airline/Carrier")
    .when(col("CANCELLATION_REASON") == "B", "Weather")
    .when(col("CANCELLATION_REASON") == "C", "National Air System")
    .when(col("CANCELLATION_REASON") == "D", "Security")
)

### FILL NULL DELAYS

In [0]:
delay_cols = [
    "WEATHER_DELAY","AIRLINE_DELAY","AIR_SYSTEM_DELAY",
    "SECURITY_DELAY","LATE_AIRCRAFT_DELAY"
]

for c in delay_cols:
    df = df.withColumn(c, coalesce(col(c), lit(0.0)))

### Total Delays

In [0]:
df = df.withColumn(
    "total_delay_minutes",
    col("WEATHER_DELAY") + col("AIRLINE_DELAY") +
    col("AIR_SYSTEM_DELAY") + col("SECURITY_DELAY") +
    col("LATE_AIRCRAFT_DELAY")
)

### Time Anamolies

In [0]:
df = df.withColumn(
    "is_time_anomaly",
    (col("ELAPSED_TIME") <= 0) | (col("AIR_TIME") <= 0)
)

### Tail Number Fix

In [0]:
df = df.withColumn(
    "TAIL_NUMBER",
    coalesce(col("TAIL_NUMBER"), lit("UNKNOWN"))
)

### Duplicate detection

In [0]:
window_spec = Window.partitionBy(
    "FLIGHT_NUMBER","ORIGIN_AIRPORT","DESTINATION_AIRPORT","flight_date"
)


df = df.withColumn("dup_count", count("*").over(window_spec)) \
       .withColumn("is_duplicate", col("dup_count") > 1) \
       .drop("dup_count")

### Taxi + Route

In [0]:
df = df.withColumn(
    "taxi_total_minutes",
    coalesce(col("TAXI_OUT"), lit(0)) + coalesce(col("TAXI_IN"), lit(0))
)

df = df.withColumn(
    "route_id",
    concat_ws("-", col("ORIGIN_AIRPORT"), col("DESTINATION_AIRPORT"))
)

In [0]:
airlines_df.printSchema()

In [0]:
df = df.join(
    airlines_df.select(
        col("IATA_CODE").alias("airline_code"),
        col("AIRLINE").alias("airline_full_name")
    ),
    df.AIRLINE == col("airline_code"),
    "left"
).drop("airline_code")

In [0]:
airports_df.printSchema()

In [0]:
airport_full = airports_df.select(
    col("IATA_CODE").alias("airport_code"),
    col("CITY"),
    col("STATE"),
    col("LATITUDE"),
    col("LONGITUDE")
)

In [0]:
# airport_full = airports_df.select(
#     col("IATA_CODE").alias("airport_code"),
#     "CITY","STATE","LATITUDE","LONGITUDE"
# )

# # Origin
# df = df.join(
#     airport_full.withColumnRenamed("airport_code","origin_airport_code"),
#     df.ORIGIN_AIRPORT == col("origin_airport_code"),
#     "left"
# ).withColumnRenamed("CITY","origin_city") \
#  .withColumnRenamed("STATE","origin_state") \
#  .withColumnRenamed("LATITUDE","origin_lat") \
#  .withColumnRenamed("LONGITUDE","origin_lon")

# # Destination
# df = df.join(
#     airport_full.withColumnRenamed("airport_code","dest_airport_code"),
#     df.DESTINATION_AIRPORT == col("dest_airport_code"),
#     "left"
# ).withColumnRenamed("CITY","dest_city") \
#  .withColumnRenamed("STATE","dest_state") \
#  .withColumnRenamed("LATITUDE","dest_lat") \
#  .withColumnRenamed("LONGITUDE","dest_lon")

# df = df.drop("origin_airport_code","dest_airport_code")

In [0]:
airports_df.printSchema()

In [0]:
airport_full = airports_df.select(
    col("IATA_CODE").alias("airport_code"),
    "CITY","STATE","LATITUDE","LONGITUDE"
)

In [0]:
df.printSchema()

In [0]:
df = df.join(
    airport_full.withColumnRenamed("airport_code","origin_airport_code"),
    df.ORIGIN_AIRPORT == col("origin_airport_code"),
    "left"
).withColumnRenamed("CITY","origin_city") \
 .withColumnRenamed("STATE","origin_state") \
 .withColumnRenamed("LATITUDE","origin_lat") \
 .withColumnRenamed("LONGITUDE","origin_lon")

In [0]:
airport_full.printSchema()

In [0]:


# Origin


# Destination
df = df.join(
    airport_full.withColumnRenamed("airport_code","dest_airport_code"),
    df.DESTINATION_AIRPORT == col("dest_airport_code"),
    "left"
).withColumnRenamed("CITY","dest_city") \
 .withColumnRenamed("STATE","dest_state") \
 .withColumnRenamed("LATITUDE","dest_lat") \
 .withColumnRenamed("LONGITUDE","dest_lon")

df = df.drop("origin_airport_code","dest_airport_code")

### WRITE SILVER FLIGHTS

In [0]:
CHECKPOINTS_PATH = "/Volumes/Skylens_macro/team1/checkpoints/silver"   

querey=(df.writeStream.format("delta") \
    .outputMode("append")\
        .option('checkpointLocation',f"{CHECKPOINTS_PATH}/flights/checkpoints/")
    
               .option("mergeSchema", "true")\
                   .partitionBy("flight_date") \
        .trigger(availableNow=True)
    .toTable("silver_flights")

)

### SILVER AIRLINES

In [0]:
airlines_df.printSchema()

In [0]:
airlines_df.show(truncate=False)

In [0]:
silver_airlines = airlines_df.select(
    col("IATA_CODE").alias("airline_code"),
    col("AIRLINE")
).dropDuplicates()

silver_airlines.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_airlines")


### SILVER AIRPORTS

In [0]:
silver_airports = airports_df.select(
    col("IATA_CODE").alias("airport_code"),
    col("AIRPORT").alias("airport_name"),
    "CITY","STATE","COUNTRY","LATITUDE","LONGITUDE"
).dropDuplicates()

silver_airports.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_airports")

In [0]:
%sql
SELECT COUNT(*) FROM silver_flights;
SELECT COUNT(*) FROM silver_airlines;
SELECT COUNT(*) FROM silver_airports;